# How to Choose an Encoder

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/choose_an_encoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Fchoose_an_encoder.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/choose_an_encoder.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


Three tools for picking the right encoder before committing to a training run:
- `recommend()` — best encoder for a task + qubit budget
- `suggest_qubits()` — how many qubits your dataset actually needs
- `compare_encodings()` — side-by-side cost table for several encoders

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning
from quprep.core.dataset import Dataset

print(f"quprep {qd.__version__}")

rng = np.random.default_rng(42)
X = rng.uniform(0, np.pi, (80, 6))
dataset = Dataset(data=X)

## 1. `recommend()` — best encoder for your task

In [ ]:
rec = qd.recommend(dataset, task="classification", qubits=6)

print(f"Recommended encoder : {rec.method}")
print(f"Score               : {rec.score:.3f}")
print(f"NISQ safe           : {rec.nisq_safe}")
print(f"Reason              : {rec.reason}")
print(f"Alternatives        : {rec.alternatives}")

## 2. `suggest_qubits()` — qubit budget

In [ ]:
sq = qd.suggest_qubits(dataset, task="classification", max_qubits=8)

print(f"Suggested qubits : {sq.n_qubits}")
print(f"NISQ safe        : {sq.nisq_safe}")
print(f"Encoding hint    : {sq.encoding_hint}")
print(f"Reasoning        : {sq.reasoning}")

## 3. `compare_encodings()` — side-by-side cost table

In [ ]:
comparison = qd.compare_encodings(
    dataset,
    include=["angle", "entangled_angle", "iqp", "amplitude"],
    task="classification",
    qubits=6,
)

print(f"{'Encoding':<20} {'Qubits':>6} {'Depth':>6} {'2Q gates':>8} {'NISQ safe':>10}")
print("-" * 55)
for row in comparison.rows:
    print(f"{row.encoding:<20} {row.n_qubits:>6} {row.circuit_depth:>6} "
          f"{row.two_qubit_gates:>8} {str(row.nisq_safe):>10}")
print(f"\nBest choice : {comparison.best().encoding}")

## 4. Apply the recommendation via `suggest_pipeline()`

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    pipeline = qd.suggest_pipeline(dataset, task="classification", qubits=6).build()
    result = pipeline.fit_transform(dataset)

print(f"Circuits : {len(result.encoded)}")
print(f"Qubits   : {result.encoded[0].metadata.get('n_qubits')}")